# Phase B6 — Valence denoising (neutral-corpus PCA)

The warmth and competence direction vectors share a common **valence** component
(cos ≈ 0.75 for Gemma-3-12B): stories that portray someone positively — whether
warmly or competently — land in a similar region of the residual stream. This
notebook removes that shared component by projecting out the top principal
components of a **socially neutral** corpus (Wikipedia intros), following the
Anthropic emotion-concepts recipe.

**What it tests.** If, after denoising, the two axes separate (cos drops toward 0)
*and* each still classifies its own axis while leaking less onto the other, we have
clean, separable warmth/competence directions. If denoising instead *destroys* the
signal, that tells us the contrast we were reading was mostly valence — also a
reportable finding.

## Where to run

**JupyterHub research space, Full GPU (80 GB)**, kernel started in the repo root.
Step 2 loads Gemma on the GPU, so this does **not** run on your laptop. Step 1
(building the neutral corpus) needs internet: if this kernel has no outbound
network, build the corpus once on the **SCCKN login node** (one command, shown
below) and re-run from Step 2.

**Prerequisites:** the Phase-4 vectors in `data/processed/<VECTORS_SUBDIR>/`
(`warmth_vec.npy`, `competence_vec.npy`, `X_<condition>.npy`, `meta.json`) and
`pip install datasets scikit-learn`.

In [6]:
#!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124 --quiet
#pip install transformer-lens

In [1]:
#import torch
#print(torch.cuda.is_available())   

True


In [1]:
import os
os.chdir("/home/jovyan/normalcy-axis")   # adjust path if different

In [8]:
# --- setup (same pattern as notebooks 06 / 07) ---
import sys, json, time
from pathlib import Path
from dataclasses import replace
import numpy as np
import torch

#PYTHONPATH=. python scripts/build_neutral_corpus.py --config config/config.yaml

REPO = Path.cwd()
assert (REPO / "src").exists(), "Start this notebook from the repository root (normalcy-axis/)."
sys.path.insert(0, str(REPO))

from src.utils.config import load_config
from src.utils.hooks import residual_hook_name
from src.utils.model_loader import load_hooked_model

# Choose which model's vectors to denoise. Default = 12B baseline.
# For the scale check set VECTORS_SUBDIR = "concept_vectors_gemma3_27b".
VECTORS_SUBDIR = "concept_vectors_gemma3_27b"

cfg = load_config("config/config.yaml")
vdir = Path(cfg.paths.processed) / VECTORS_SUBDIR
meta = json.loads((vdir / "meta.json").read_text(encoding="utf-8"))
LAYER = int(meta["probe_layer"]); HOOK = residual_hook_name(LAYER)
print("model:", meta["model"], "| probe layer:", LAYER, "| hook:", HOOK)
print("neutral config:", cfg.neutral.n_texts, "texts,",
      cfg.neutral.min_words, "-", cfg.neutral.max_words, "words,",
      "variance threshold", cfg.neutral.variance_threshold)

model: google/gemma-3-27b-it | probe layer: 40 | hook: blocks.40.hook_resid_post
neutral config: 1500 texts, 90 - 200 words, variance threshold 0.5


## Step 1 — Build the neutral corpus (needs internet)

Streams Wikipedia intros, length-matches them to the concept stories, and drops
valence-laden articles. The corpus is **model-agnostic** — build it once and reuse
for every model. If this kernel has no internet, run instead on the login node:

```bash
PYTHONPATH=. python scripts/build_neutral_corpus.py --config config/config.yaml
```

and then skip to Step 2.

In [9]:
corpus_path = Path(cfg.neutral.corpus_path)
if corpus_path.exists():
    n = sum(1 for _ in corpus_path.open(encoding="utf-8"))
    print(f"neutral corpus already present: {corpus_path} ({n} texts) — skipping build")
else:
    print("building neutral corpus (requires internet)...")
    try:
        from scripts.build_neutral_corpus import build
        build(cfg)
    except Exception as e:
        print("\nBUILD FAILED:", repr(e))
        print("If this is a network error, build on the SCCKN login node instead:")
        print("  PYTHONPATH=. python scripts/build_neutral_corpus.py --config config/config.yaml")
        raise

neutral corpus already present: data/stimuli/neutral_corpus.jsonl (1500 texts) — skipping build


## Step 2 — Load the model and extract neutral activations (GPU)

We extract residual-stream activations for the neutral texts at the **same layer
and pooling rule** used for the concept stories, so the PCA lives in the same
space as the warmth/competence vectors. Activations are model-specific, so this
step is repeated per model (saved inside the model's own vectors folder).

In [10]:
from huggingface_hub import login
login()  # paste your HF token when prompted (Gemma is gated)

In [11]:
# load the model named in meta.json so 12B / 27B both work
cfg = replace(cfg, model=replace(cfg.model, name=str(meta["model"])))
print("loading", cfg.model.name, "... (a few minutes on first load)")
model = load_hooked_model(cfg); model.eval()
print("loaded. n_layers =", model.cfg.n_layers, "| d_model =", model.cfg.d_model)

loading google/gemma-3-27b-it ... (a few minutes on first load)


Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-3-27b-it into HookedTransformer
loaded. n_layers = 62 | d_model = 5376


In [12]:
# reuse the identical extraction loop from the pipeline (no logic drift)
from src.extract_vectors import extract_activations

def load_neutral(path):
    texts = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                texts.append(json.loads(line)["text"])
    return texts

texts = load_neutral(cfg.neutral.corpus_path)
print(len(texts), "neutral texts | extracting at", HOOK, "...")
torch.manual_seed(cfg.probing.seed); np.random.seed(cfg.probing.seed)
with torch.no_grad():
    Xn = extract_activations(model, texts, HOOK, cfg.probing.start_token)
Xn = np.asarray(Xn.numpy() if hasattr(Xn, "numpy") else Xn, dtype=np.float32)
np.save(vdir / "X_neutral.npy", Xn)
(vdir / "neutral_meta.json").write_text(json.dumps({
    "n_neutral": len(texts), "probe_layer": LAYER, "d_model": int(model.cfg.d_model),
    "start_token": cfg.probing.start_token, "corpus": str(cfg.neutral.corpus_path),
    "seed": cfg.probing.seed, "timestamp": int(time.time())}, indent=2), encoding="utf-8")
print("saved X_neutral.npy", Xn.shape, "->", vdir)

1500 neutral texts | extracting at blocks.40.hook_resid_post ...
  [extract] 10/1500 done
  [extract] 20/1500 done
  [extract] 30/1500 done
  [extract] 40/1500 done
  [extract] 50/1500 done
  [extract] 60/1500 done
  [extract] 70/1500 done
  [extract] 80/1500 done
  [extract] 90/1500 done
  [extract] 100/1500 done
  [extract] 110/1500 done
  [extract] 120/1500 done
  [extract] 130/1500 done
  [extract] 140/1500 done
  [extract] 150/1500 done
  [extract] 160/1500 done
  [extract] 170/1500 done
  [extract] 180/1500 done
  [extract] 190/1500 done
  [extract] 200/1500 done
  [extract] 210/1500 done
  [extract] 220/1500 done
  [extract] 230/1500 done
  [extract] 240/1500 done
  [extract] 250/1500 done
  [extract] 260/1500 done
  [extract] 270/1500 done
  [extract] 280/1500 done
  [extract] 290/1500 done
  [extract] 300/1500 done
  [extract] 310/1500 done
  [extract] 320/1500 done
  [extract] 330/1500 done
  [extract] 340/1500 done
  [extract] 350/1500 done
  [extract] 360/1500 done
  [extra

## Step 3 — PCA denoise and report before / after

Fit PCA on the neutral activations, keep the top components covering ≥ 50% of
neutral variance, and project them out of both direction vectors. We then report,
before and after:
- **cos(warmth, competence)** — should drop sharply if the overlap was valence;
- **Cohen's d** per axis — should *survive* if the concept signal is real;
- **warmth-on-competence leak** — the warmth vector separating the competence
  pairs; should drop toward 0.

In [13]:
# reuse the denoising helpers from src so the notebook matches the script exactly
from src.denoise_vectors import project_out, select_k, cohens_d, cosine, CONDITIONS
from sklearn.decomposition import PCA

warm = np.load(vdir / "warmth_vec.npy"); comp = np.load(vdir / "competence_vec.npy")
cond = {c: np.load(vdir / f"X_{c}.npy") for c in CONDITIONS}
thr = cfg.neutral.variance_threshold

pca = PCA(random_state=cfg.probing.seed).fit(Xn)
k = select_k(pca.explained_variance_ratio_, thr)
comps = pca.components_[:k]
kept = float(pca.explained_variance_ratio_[:k].sum())
print(f"[pca] neutral n={Xn.shape[0]} d={Xn.shape[1]} -> k={k} components cover {kept:.3f} variance")

warm_d = project_out(warm, comps); comp_d = project_out(comp, comps)

def report(tag, wv, cv):
    cw = cosine(wv, cv)
    d_w = cohens_d(cond["high_warmth"] @ wv, cond["low_warmth"] @ wv)
    d_c = cohens_d(cond["high_competence"] @ cv, cond["low_competence"] @ cv)
    leak = cohens_d(cond["high_competence"] @ wv, cond["low_competence"] @ wv)
    print(f"  [{tag:8}] cos(w,c)={cw:+.3f}   d_warmth={d_w:5.2f}   d_competence={d_c:5.2f}   warmth-on-competence(leak)={leak:5.2f}")
    return cw

print("\nBEFORE vs AFTER denoising:")
cos_before = report("raw", warm, comp)
cos_after = report("denoised", warm_d, comp_d)

[pca] neutral n=1500 d=5376 -> k=43 components cover 0.502 variance

BEFORE vs AFTER denoising:
  [raw     ] cos(w,c)=+0.708   d_warmth= 2.92   d_competence= 3.24   warmth-on-competence(leak)= 2.29
  [denoised] cos(w,c)=+0.487   d_warmth= 8.77   d_competence= 9.50   warmth-on-competence(leak)= 4.92


In [14]:
# save denoised vectors + summary
np.savez(vdir / "concept_vectors_denoised.npz",
         warmth=warm_d, competence=comp_d, neutral_pca_components=comps,
         k=k, variance_threshold=thr, cosine_before=cos_before, cosine_after=cos_after)
json.dump({"k": int(k), "variance_kept": kept,
           "cosine_before": cos_before, "cosine_after": cos_after},
          (vdir / "denoise_summary.json").open("w"), indent=2)
print(f"saved concept_vectors_denoised.npz  (k={k}, cos {cos_before:+.3f} -> {cos_after:+.3f})")

saved concept_vectors_denoised.npz  (k=43, cos +0.708 -> +0.487)


## Step 4 — How to read this, and what comes next

**Interpretation grid:**

| cos(w,c) after | d_warmth / d_competence after | leak after | Meaning |
|---|---|---|---|
| drops toward 0 | stay high | drops toward 0 | **Best case:** clean, separable warmth/competence axes — the overlap *was* valence. |
| drops | collapse too | drops | The contrast we read was mostly valence; report honestly. |
| stays high | — | — | The shared component is not captured by neutral-corpus PCA; rethink the neutral set or threshold. |

**Next steps once denoised vectors exist:**
1. Refresh Figure 1 (joint density) and Figure 4 (geometry) using the denoised
   vectors to show the reduced diagonal.
2. **Re-run causal steering (notebooks 06 / 07) with the denoised vectors** — the
   key test is whether warmth steering now moves the warmth judgement but *not*
   competence (axis-specific causality). Use the **local** steering regime
   (`[-0.1 … 0.1]`, see `docs/robustness_audit.md` A1).
3. Repeat this notebook at 27B (`VECTORS_SUBDIR = "concept_vectors_gemma3_27b"`).
4. (Optional, decision D5) sweep the variance threshold (0.40 / 0.50 / 0.60) and
   report sensitivity in the supplement.